# RecSys МАГОЛЕГО, ФКН ВШЭ

## Домашняя работа 2 (Часть 2): GCN

### Оценивание и штрафы
Всего заданий: **5**, максимальная оценка — **6 баллов (+2.0 бонус)**.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом пользоваться запрещается,даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке (например, лишние циклы, `np.vectorize`, `np.apply_along_axis` и так далее). Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных языковых моделей разрешено только в случае явного указания на это. Необходимо прописать (в соответствующих пунктах, где использовались, либо в начале/конце работы):

- какая языковая модель использовалась
- какие использовались промпты и в каких частях работы
- с какими сложностями вы столкнулись при использовании генеративных моделей, с чем они помогли больше всего
  
Copy-paste ответа генеративной модели запрещается (кроме графиков, но все равно нужно явно прописывать использование)

**Дедлайн: 15.03.2026 23:59 (по МСК)**

**Сдавать сюда: [Классрум](https://classroom.google.com/c/ODQzNjI1ODIzMDEy/a/ODQ2ODc0NzExMTI5/details)**

### О задании
В данном домашнем задании вы реализуете алгоритмы рекомендаций на основе графовых нейросетей для обучения эмбеддингов на датасете Yahoo! Movies.

P.S Пожалуйста, аккуратно оформляйте графики, ориентироваться можно на [это](https://github.com/esokolov/ml-course-hse/blob/master/2022-fall/seminars/sem02-charts.ipynb). У графиков обязательно должно быть:

- Название
- Подписанные оси
- Легенда, если необходимо (например, если несколько графиков на одном рисунке)
- Все должно быть четко видно и ничего не сливаться
- За некрасивые графики можем снизить баллы

In [ ]:
# !pip install torch_geometric

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_geometric

## Структура датасета

Датасет можете скачать по [ссылке](https://drive.google.com/file/d/1PsAL83MQnvQuTpjrNXs8CiPpOeSGcUL-/view?usp=share_link)

Датасет состоит из 6 основных файлов. Все данные представлены в текстовом формате, где колонки разделены символом табуляции `\t`.

### 1. Оценки пользователей
В этих файлах содержатся оценки фильмов пользователей
-  **Файлы:** `ydata-ymovies-user-movie-ratings-train-v1_0` и `ydata-ymovies-user-movie-ratings-test-v1_0`
- **Поля:**
  * User ID
  * Movie ID
  * Rating_13: от 1 (F) до 13 (A+)
  * Rating_5: упрощенная шкала от 1 до 5



### 2. Демография пользователей
Информация про аудиторию

- **Файл:** `ydata-ymovies-user-demographics-v1_0`
- **Поля:** 
  * User ID
  * Year of birth
  * Sex (`m`/`f`)

### 3. Описание фильмов
Метаданные фильмов
* **Файл:** `ydata-ymovies-movie-content-descr-v1_0`
* **Что внутри:** Название, синопсис, жанры, режиссеры, актеры, количество наград, средняя оценка критиков и даже ссылки на постеры.
* **Важно:** Если данных нет, стоит заглушка `\N`. Если в поле несколько значений (например, список актеров), они разделены символом `|`.

### 4. Списки соответствия 
Файлы `mapping-to-movielens` и `mapping-to-eachmovie` позволяют связать ID фильмов Yahoo с другими популярными датасетами (MovieLens и EachMovie). Это полезно, если вы хотите объединить данные из разных источников


### GCN

В этой части мы будем использовать строить эмбеддинги пользователей и фильмов с помощью графа Лапласиана, чтобы потом на этих эмбеддингах обучать supervised модель, которая уже будет рекомендовать фильмы

**Задание 0 (1 балл):** Как и в первой части ДЗ-2, вам необходимо построить двудольный граф взаимодействий `user-item` (берем оценки $\geqslant 4$ по `rating_5`). Убедитесь, что индексы узлов перекодированы в непрерывный диапозон от $0$ до $N-1$, где $N$ - количество пользователей и фильмов. 

In [ ]:
train_ratings = pd.read_csv(
    "dataset/ydata-ymovies-user-movie-ratings-train-v1_0.txt",
    sep="\t",
    header=None,
    names=["user_id", "movie_id", "rating_13", "rating_5"],
)

test_ratings = pd.read_csv(
    "dataset/ydata-ymovies-user-movie-ratings-test-v1_0.txt",
    sep="\t",
    header=None,
    names=["user_id", "movie_id", "rating_13", "rating_5"],
)

In [ ]:
# Your code her (・・? )

**Задание 1 (2 балла):** Обучать GCN будем для предсказаний связей в графе (Link Prediction)

Реализация модели:
- Инициализируйте случайные обучаемые эмбеддинги для пользователей и айтемов
- Реализуйте базовую графовую модель (простой Message Passing), которая обновляет эмбеддинги на основе структуры графа. Используйте матрицы весов и функции активации на каждом слое
- В качестве декодера используйте скалярное произведение: предсказание для пары $u$ и $i$ вычисляется как $E_u \cdot E_i$. Также можете использовать идеи из 1 части ДЗ-2

In [ ]:
# Your code her (・・? )

**Задание 2 (2 балла):**  Реализуйте генератор негативных примеров (Negative Sampler) на этапе обучения. На каждое реальное взаимодействие (позитивное ребро) модель должна видеть одно или несколько несуществующих взаимодействий (негативные ребра). Можете взять из 1 части ДЗ-2

Используйте функцию потерь для ранжирования: BPR Loss (Bayesian Personalized Ranking) или BCE Loss (Binary Cross-Entropy) и обучите свою модель

In [ ]:
# Your code her (・・? )

**Задание 3 (1 балл):** Теперь протестируйте свою модель на тестовой выборке с метриками из ДЗ-1

In [ ]:
# Your code her (・・? )

**Задание 4 (2 балла):** Классический GCN часто страдает от проблемы пересглаживания (over-smoothing) и долго обучается на графах взаимодействий

Реализуйте архитектуру LightGCN, убрав из базовой модели трансформацию признаков (матрицы весов) и нелинейности. Оставьте только взвешенную агрегацию эмбеддингов соседей

Обучите и сравните на тестовой выборке в одинаковых условиях (random seed, lr,  batch_size, etc...) с моделью из задания 1

In [ ]:
# Your code her (・・? )

**Бонусное задание 2 (2 балла):** Так же, как и в 1 части, давайте добавим признаки пользователей и фильмов. Вам нужно будет продумать преодобработку данных и как лучше их подать модели. Сравните, насколько улучшились или ухудшились рекомендации на тестовой выборке



In [ ]:
user_features = pd.read_csv(
    "dataset/ydata-ymovies-user-demographics-v1_0.txt",
    sep="\t",
    header=None,
    names=["user_id", "year", "gender"],
)
user_features.head()

In [ ]:
cols = [
    "movie_id",
    "title",
    "synopsis",
    "running_time",
    "mpaa_rating",
    "mpaa_reasons",
    "release_date_txt",
    "release_date",
    "distributor",
    "poster_url",
    "genres",
    "directors",
    "director_ids",
    "crew",
    "crew_ids",
    "crew_types",
    "actors",
    "actor_ids",
    "avg_critic_rating",
    "num_critic_ratings",
    "awards_won",
    "awards_nominated",
    "awards_won_list",
    "awards_nominated_list",
    "movie_mom_rating",
    "movie_mom_review",
    "review_summaries",
    "review_owners",
    "trailer_captions",
    "greg_preview_url",
    "dvd_review_url",
    "gnpp",
    "avg_user_rating",
    "num_user_ratings",
]

movie_features = pd.read_csv(
    "dataset/ydata-ymovies-movie-content-descr-v1_0.txt",
    sep="\t",
    header=None,
    names=cols,
    na_values="\\N",
    encoding="latin-1",
)
movie_features.head()

In [ ]:
# Your code her (・・? )

## Conclusion

Опишите выводы ваших экспериментов:
